# Structural Design Patterns

## Adapter

In [1]:
# Adapter allows two incompatible interfaces to work together without modifying either one.

In [ ]:
class PaymentProcessor:
    def pay(self, amount):
        print(f"Paying ₹{amount}")

def checkout(payment_processor):
    payment_processor.pay(1000)


class RazorpayAPI:
    def make_payment(self, amount):
        print(f"Razorpay payment: ₹{amount}")

In [ ]:
class RazorpayAdapter:

    def __init__(self, razorpay):
        self.razorpay = razorpay

    def pay(self, amount):
        self.razorpay.make_payment(amount)

razorpay = RazorpayAPI()

payment = RazorpayAdapter(razorpay)

payment.pay(1000)

In [2]:
# Your Application
#        |
#        | expects
#        ↓
#      pay()
#        X
#        |
#        | provides
#        ↓
# make_payment()
# Third Party

In [3]:
# Application
#     |
#     | expected interface
#     ↓
#   Adapter
#     |
#     | converts request
#     ↓
# Third-party system

In [ ]:
from abc import ABC, abstractmethod


class PaymentProcessor(ABC):

    @abstractmethod
    def pay(self, amount):
        pass

def checkout(payment_processor: PaymentProcessor):
    payment_processor.pay(1000)

class StripePayment(PaymentProcessor):
    def pay(self, amount):
        print(f"Stripe payment: ₹{amount}")


class RazorpayAPI:
    def make_payment(self, amount):
        print(f"Razorpay payment: ₹{amount}")

class RazorpayAdapter(PaymentProcessor):
    def __init__(self, razorpay):
        self.razorpay = razorpay

    def pay(self, amount):
        self.razorpay.make_payment(amount)



razorpay = RazorpayAPI()
payment_processor = RazorpayAdapter(razorpay)
checkout(payment_processor)

In [ ]:
class LLM:

    def generate(self, prompt):
        pass

def run_agent(llm: LLM, prompt):
    return llm.generate(prompt)

class OpenAIClient:

    def chat_completion(self, prompt):
        return "OpenAI response"

class GeminiClient:

    def generate_content(self, prompt):
        return "Gemini response"


class NvidiaClient:

    def chat(self, prompt):
        return "NVIDIA response"

In [ ]:
class OpenAIAdapter(LLM):

    def __init__(self, client):
        self.client = client

    def generate(self, prompt):
        return self.client.chat_completion(prompt)


class GeminiAdapter(LLM):

    def __init__(self, client):
        self.client = client

    def generate(self, prompt):
        return self.client.generate_content(prompt)


class NvidiaAdapter(LLM):

    def __init__(self, client):
        self.client = client

    def generate(self, prompt):
        return self.client.chat(prompt)


In [ ]:
run_agent(OpenAIAdapter(openai_client), "Hello")
run_agent(OpenAIAdapter(openai_client), "Hello")
run_agent(NvidiaAdapter(nvidia_client), "Hello")

                    Application
                        |
                        ↓
                       Agent
                        |
                        ↓
                       LLM
                        ↑
                        |
                   LLM Factory
                        |
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
       OpenAI         Gemini        NVIDIA
       Adapter        Adapter       Adapter
          ↓             ↓             ↓
     OpenAI API     Gemini API    NVIDIA API

In [ ]:
# project/
# │
# ├── agents/
# │   └── research_agent.py
# │
# ├── llm/
# │   ├── interface.py
# │   ├── factory.py
# │   │
# │   └── adapters/
# │       ├── openai_adapter.py
# │       ├── gemini_adapter.py
# │       └── nvidia_adapter.py
# │
# ├── tools/
# │
# ├── config/
# │
# └── runner/

In [ ]:
# interface.py

class LLM:

    def generate(self, prompt):
        raise NotImplementedError

In [ ]:
# nvidia_adapter.py

class NvidiaAdapter(LLM):

    def __init__(self, client):
        self.client = client

    def generate(self, prompt):
        return self.client.chat(prompt)

In [ ]:
# research_agent.py

class ResearchAgent:

    def __init__(self, llm):
        self.llm = llm

    def run(self, prompt):
        return self.llm.generate(prompt)

## decorators

In [ ]:
# Add new behavior to an existing object without modifying its original class.

In [ ]:
class EmailNotification:
    def send(self, message):
        print(f"Sending email: {message}")

notification = EmailNotification()
notification.send("Order placed")

Sending email: Order placed


In [ ]:
class EmailNotification:

    def send(self, message):

        # logging
        print("Logging...")

        # authentication
        print("Checking authentication...")

        # encryption
        print("Encrypting...")

        # actual operation
        print(f"Sending email: {message}")

        # metrics
        print("Recording metrics...")

In [ ]:
from abc import ABC, abstractmethod


class Notification(ABC):
    @abstractmethod
    def send(self, message):
        pass


class EmailNotification(Notification):
    def send(self, message):
        print(f"Sending email: {message}")

class NotificationDecorator(Notification):
    def __init__(self, notification):
        self.notification = notification

    def send(self, message):
        self.notification.send(message)

class LoggingDecorator(NotificationDecorator):
    def send(self, message):

        print("LOG: Sending notification")

        self.notification.send(message)


class AuthenticationDecorator(NotificationDecorator):
    def send(self, message):

        print("Checking authentication")

        self.notification.send(message)


notification = EmailNotification()
notification = LoggingDecorator(notification)
notification = AuthenticationDecorator(notification)

notification.send("Order placed")

Checking authentication
LOG: Sending notification
Sending email: Order placed


In [ ]:
# notification.send()
#        ↓
# AuthenticationDecorator
#        ↓
# LoggingDecorator
#        ↓
# EmailNotification

In [ ]:
# EmailNotification
#     ├── logging
#     ├── authentication
#     ├── retry
#     ├── metrics
#     └── encryption

In [ ]:
# LoggingDecorator       → logging
# AuthenticationDecorator → authentication
# RetryDecorator          → retry
# MetricsDecorator        → metrics
# EmailNotification       → sending email

In [ ]:
# 1. This is the decorator function
def my_decorator(original_function):
    def wrapper():
        print("Something is happening BEFORE the original function runs.")
        original_function()  # Running your actual function
        print("Something is happening AFTER the original function runs.")
    return wrapper  # Returns the modified function ready to use


In [11]:
def log_activity(func):
    def wrapper():
        print(f"[LOG]: Starting {func.__name__}...")
        func()
        print(f"[LOG]: Finished {func.__name__}...\n")
    return wrapper

@log_activity
def say_hello():
    print("Hello, World!")

@log_activity
def say_goodbye():
    print("Goodbye, everyone!")

# Call the functions normally
say_hello()
say_goodbye()


[LOG]: Starting say_hello...
Hello, World!
[LOG]: Finished say_hello...

[LOG]: Starting say_goodbye...
Goodbye, everyone!
[LOG]: Finished say_goodbye...



In [ ]:
def double_result(func):
    # *args and **kwargs accept any number of inputs passed to the function
    def wrapper(*args, **kwargs):
        original_result = func(*args, **kwargs)
        modified_result = original_result * 2
        return modified_result
    return wrapper

@double_result
def add_numbers(a, b):
    return a + b

# 5 + 3 = 8, then the decorator doubles it to 16
final_score = add_numbers(5, 3)
print(f"Final Score: {final_score}") 



Final Score: 16


In [13]:
def add_all_numbers(*args):
    # args is treated as a tuple: (1, 2, 3, 4, 5)
    print(f"The packed tuple looks like: {args}")
    return sum(args)

# You can pass 3 numbers
print(f"Total: {add_all_numbers(10, 20, 30)}\n")

# Or you can pass 5 numbers using the exact same function
print(f"Total: {add_all_numbers(1, 2, 3, 4, 5)}")


The packed tuple looks like: (10, 20, 30)
Total: 60

The packed tuple looks like: (1, 2, 3, 4, 5)
Total: 15


In [15]:
def print_user_profile(**kwargs):
    # kwargs is treated as a dictionary: {'name': 'Alice', 'role': 'Admin'}
    print(f"The packed dictionary looks like: {kwargs}")
    
    for key, value in kwargs.items():
        print(f"{key.capitalize()}: {value}")

    return kwargs

# Pass any combination of key-value pairs
print_user_profile(name="Alice", role="Admin", location="India")


The packed dictionary looks like: {'name': 'Alice', 'role': 'Admin', 'location': 'India'}
Name: Alice
Role: Admin
Location: India


{'name': 'Alice', 'role': 'Admin', 'location': 'India'}

In [52]:
def double_result(func):
    # *args and **kwargs accept any number of inputs passed to the function
    def wrapper(*args, **kwargs):
        print(*args)
        # print(**kwargs)
        original_result = func(*args, **kwargs)
        modified_result = original_result * 2
        return modified_result
    return wrapper

@double_result
def add_numbers(a, b, x,y):
    k = x + y
    return a + b + k

final_score = add_numbers(5, 3, x= 1,y = 3)
print(f"Final Score: {final_score}") 


5 3
Final Score: 24


In [ ]:
def add_numbers(*args, **kwargs):
    print(args)
    print(kwargs)
    
    k = sum(kwargs.values())
    return sum(args) + k

def add_numbers(a, b, x,y):
    k = x + y
    return a + b + k


final_score = add_numbers(5, 3, x=1, y=3)
print(f"Final Score: {final_score}") 

Final Score: 12


In [50]:

def add_numbers(a, b, x,y):
    k = x + y
    return a + b + k

def wrapper(*args, **kwargs):
    final_score = add_numbers(*args, **kwargs)
    print(add_numbers)
    print(f"Final Score: {final_score}")

wrapper(5, 3, x=1, y=3)

<function add_numbers at 0x000001D62EC06160>
Final Score: 12


In [54]:
class LLM:

    def generate(self, prompt):
        return "LLM response"

llm = LLM()

class LLM:
    def generate(self, prompt):
        # logging
        # metrics
        # retry
        # cache
        # tracing
        # actual call
        pass

In [ ]:
class LoggingLLM:

    def __init__(self, llm):
        self.llm = llm

    def generate(self, prompt):

        print(f"LLM request: {prompt}")

        response = self.llm.generate(prompt)

        print(f"LLM response: {response}")

        return response

class RetryLLM:

    def __init__(self, llm):
        self.llm = llm

    def generate(self, prompt):

        for attempt in range(3):

            try:
                return self.llm.generate(prompt)

            except Exception as e:

                print(f"Attempt {attempt + 1} failed")

        raise RuntimeError("LLM failed")


llm = LLM()

llm = RetryLLM(llm)
llm = LoggingLLM(llm)

In [58]:
# Agent
#  ↓
# LoggingDecorator
#  ↓
# RetryDecorator
#  ↓
# NvidiaAdapter
#  ↓
# NVIDIA Client



#                 Factory
#                    ↓
#              Create NVIDIA LLM
#                    ↓
#               Adapter
#                    ↓
#               Retry Decorator
#                    ↓
#              Logging Decorator
#                    ↓
#              Metrics Decorator
#                    ↓
#                 Agent

In [ ]:
from abc import ABC, abstractmethod


# Component
class Notification(ABC):

    @abstractmethod
    def send(self, message):
        pass


# Concrete Component
class EmailNotification(Notification):

    def send(self, message):
        print(f"Sending email: {message}")


# Base Decorator
class NotificationDecorator(Notification):

    def __init__(self, notification):
        self.notification = notification

    def send(self, message):
        self.notification.send(message)


# Concrete Decorator 1
class LoggingDecorator(NotificationDecorator):

    def send(self, message):

        print("LOG: Notification started")

        self.notification.send(message)


# Concrete Decorator 2
class AuthenticationDecorator(NotificationDecorator):

    def send(self, message):

        print("AUTH: Checking authentication")

        self.notification.send(message)


# Concrete Decorator 3
class MetricsDecorator(NotificationDecorator):

    def send(self, message):

        print("METRICS: Recording metrics")

        self.notification.send(message)


# Create original object
notification = EmailNotification()

# Add behavior
notification = LoggingDecorator(notification)
notification = AuthenticationDecorator(notification)
notification = MetricsDecorator(notification)

# Use it
notification.send("Order placed")

In [59]:
# ┌───────────────────────────────┐
# │       Metrics Decorator       │
# │  ┌─────────────────────────┐  │
# │  │    Retry Decorator      │  │
# │  │ ┌─────────────────────┐ │  │
# │  │ │ Logging Decorator   │ │  │
# │  │ │ ┌─────────────────┐ │ │  │
# │  │ │ │   Real Object   │ │ │  │
# │  │ │ └─────────────────┘ │ │  │
# │  │ └─────────────────────┘ │  │
# │  └─────────────────────────┘  │
# └───────────────────────────────┘

## facade

In [ ]:
# Facade provides a simple interface to a complex subsystem.

# inventory.check_stock()
# payment.validate_card()
# payment.process_payment()
# order_db.create_order()
# shipping.create_shipment()
# notification.send_email()

def place_order(product, customer):
    inventory.check_stock(product)
    payment.validate_card(customer)
    payment.process_payment(customer)
    order_db.create_order(product, customer)
    shipping.create_shipment(product, customer)
    notification.send_email(customer)

def place_order(...):

    inventory.check_stock()

    fraud.detect()

    tax.calculate()

    coupon.validate()

    payment.validate()

    payment.process()

    order_db.create()

    inventory.reserve()

    shipping.create()

    analytics.track()

    audit.log()

    notification.send()

In [60]:
# Application
#     ↓
# OrderFacade
#     ↓
#  ┌───────────────┐
#  │ Inventory     │
#  │ Payment       │
#  │ Shipping      │
#  │ Database      │
#  │ Notification  │
#  └───────────────┘

In [61]:
class InventoryService:

    def check_stock(self, product):
        print(f"Checking stock for {product}")
        return True


class PaymentService:

    def process_payment(self, amount):
        print(f"Processing payment: ₹{amount}")
        return True

class ShippingService:

    def create_shipment(self, product):
        print(f"Creating shipment for {product}")

class NotificationService:

    def send_confirmation(self):
        print("Sending order confirmation")

In [62]:
# without fasasd

inventory = InventoryService()
payment = PaymentService()
shipping = ShippingService()
notification = NotificationService()

if inventory.check_stock("Laptop"):

    if payment.process_payment(80000):

        shipping.create_shipment("Laptop")

        notification.send_confirmation()

Checking stock for Laptop
Processing payment: ₹80000
Creating shipment for Laptop
Sending order confirmation


In [63]:
class OrderFacade:

    def __init__(self):
        self.inventory = InventoryService()
        self.payment = PaymentService()
        self.shipping = ShippingService()
        self.notification = NotificationService()

    def place_order(self, product, amount):

        if not self.inventory.check_stock(product):
            return False

        if not self.payment.process_payment(amount):
            return False

        self.shipping.create_shipment(product)

        self.notification.send_confirmation()

        return True

order = OrderFacade()

order.place_order("Laptop", 80000)

Checking stock for Laptop
Processing payment: ₹80000
Creating shipment for Laptop
Sending order confirmation


True

In [ ]:
# #                   ┌── Inventory
# #                   │
# # Application ──────┼── Payment
# #                   │
# #                   ├── Shipping
# #                   │
# #                   └── Notification


# #                     ┌── Inventory
# #                     │
# # Application         ├── Payment
# #       ↓             │
# #  OrderFacade ───────┼── Shipping
# #                     │
# #                     └── Notification



# Facade does not mean:

# "Nobody can access the underlying classes."

# It means:

# "Provide a simpler interface for common operations."



|                      | Adapter                     | Facade                                    |
| -------------------- | --------------------------- | ----------------------------------------- |
| Main purpose         | Interface compatibility     | Simplification                            |
| Number of components | Usually one adaptee         | Usually multiple subsystems               |
| Changes interface?   | Yes                         | Provides a simpler higher-level interface |
| Main problem         | Incompatible API            | Complex subsystem                         |
| Example              | NVIDIA API → common LLM API | Agent system → simple `run()` API         |


In [ ]:
Adapter:
"These two things don't speak the same language."

Facade:
"There are too many things to deal with."

In [ ]:
query_embedding = embedding_model.embed(query)

documents = vector_db.search(query_embedding)

documents = reranker.rank(documents)

prompt = prompt_manager.build(query, documents)

response = llm.generate(prompt)

memory.save(query, response)

metrics.record(...)

In [ ]:
class ResearchAgentFacade:

    def __init__(
        self,
        llm,
        embedding_model,
        vector_db,
        reranker,
        prompt_manager,
        memory
    ):
        self.llm = llm
        self.embedding_model = embedding_model
        self.vector_db = vector_db
        self.reranker = reranker
        self.prompt_manager = prompt_manager
        self.memory = memory

    def ask(self, query):

        embedding = self.embedding_model.embed(query)

        documents = self.vector_db.search(embedding)

        documents = self.reranker.rank(documents)

        prompt = self.prompt_manager.build(
            query,
            documents
        )

        response = self.llm.generate(prompt)

        self.memory.save(query, response)

        return response

In [ ]:
# ┌──────────────────────────┐
# │       Application        │
# └────────────┬─────────────┘
#              │
#              │ ask(query)
#              ↓
# ┌──────────────────────────┐
# │    ResearchAgentFacade   │
# │                          │
# │  ask()                   │
# └────────────┬─────────────┘
#              │
#        ┌─────┼─────┬──────┬──────┐
#        ↓     ↓     ↓      ↓      ↓
#       LLM  RAG   Search  Memory  Tools

In [ ]:
Facade can become a clean boundary between:

Application

and:

Complex internal architecture

In [ ]:
class AgentFacade:

    def run(self, request):

        context = self.retrieve_context(request)

        plan = self.create_plan(request, context)

        result = self.execute_tools(plan)

        response = self.generate_response(result)

        self.record_metrics(request, response)

        return response

agent.run(request)

In [66]:
#                 Factory
#                   ↓
#             Create LLM
#                   ↓
# Application → AgentFacade
#                   ↓
#         ┌─────────┼─────────┐
#         ↓         ↓         ↓
#        LLM       RAG       Tools

                         APPLICATION
                              │
                              ↓
                         AGENT FACADE
                    "Give me an answer"
                              │
                              ↓
                       ┌─────────────┐
                       │ LLM Service │
                       └──────┬──────┘
                              │
                       DECORATOR
                              │
                    ┌─────────┴─────────┐
                    ↓                   ↓
                 Logging              Retry
                    │                   │
                    └─────────┬─────────┘
                              ↓
                           ADAPTER
                              │
                 ┌────────────┼────────────┐
                 ↓            ↓            ↓
              OpenAI        Gemini       NVIDIA

In [67]:
# Adapter    → TRANSLATE
# Decorator  → EXTEND
# Facade     → SIMPLIFY


# Provider APIs
#      ↓
#    Adapter
# "make them compatible"
#      ↓
#   Decorator
# "add logging/retry/metrics"
#      ↓
#    Facade
# "give application a simple API"
#      ↓
# Application

## flyweight (cache)


In [ ]:
# Flyweight is not exactly the same thing as a cache, although a Flyweight implementation commonly uses a cache/factory to store and reuse shared objects.
# If many objects have the same data, don't create that data repeatedly. Store it once and share it.

In [68]:
# Without Flyweight:

# Object A → "NVIDIA"
# Object B → "NVIDIA"
# Object C → "NVIDIA"
# Object D → "NVIDIA"

# Same data repeated 4 times.

              ┌──────────────┐
              │ "NVIDIA"     │
              │ Shared Object│
              └──────┬───────┘
                     ↑
              ┌──────┼───────┐
              │      │       │
             Obj A  Obj B   Obj C

In [ ]:
class Tree:

    def __init__(self, tree_type, color, texture):
        self.tree_type = tree_type
        self.color = color
        self.texture = texture

tree1 = Tree("Oak", "Green", "oak.png")
tree2 = Tree("Oak", "Green", "oak.png")
tree3 = Tree("Oak", "Green", "oak.png")

In [69]:
# Tree
# │
# ├── Intrinsic state
# │      ├── type
# │      ├── color
# │      └── texture
# │
# └── Extrinsic state
#        ├── x
#        ├── y
#        ├── size
#        └── rotation

In [70]:
class TreeType:

    def __init__(self, name, color, texture):
        self.name = name
        self.color = color
        self.texture = texture

    def draw(self, x, y):
        print(
            f"Drawing {self.name} "
            f"at ({x}, {y}) "
            f"using {self.texture}"
        )

In [71]:
class TreeTypeFactory:

    _tree_types = {}

    @classmethod
    def get_tree_type(cls, name, color, texture):

        key = (name, color, texture)

        if key not in cls._tree_types:

            print("Creating new TreeType")

            cls._tree_types[key] = TreeType(
                name,
                color,
                texture
            )

        return cls._tree_types[key]

In [72]:
tree1 = TreeTypeFactory.get_tree_type(
    "Oak",
    "Green",
    "oak.png"
)

tree2 = TreeTypeFactory.get_tree_type(
    "Oak",
    "Green",
    "oak.png"
)

tree3 = TreeTypeFactory.get_tree_type(
    "Oak",
    "Green",
    "oak.png"
)

Creating new TreeType


In [74]:
print(tree1 is tree2)
print(tree2 is tree3)

True
True


In [ ]:
# tree1
#   \
#    \
#     → Same TreeType object
#    /
#   /
# tree2

# tree3

In [77]:
class Tree:

    def __init__(self, x, y, tree_type):
        self.x = x
        self.y = y
        self.tree_type = tree_type

    def draw(self):
        self.tree_type.draw(self.x, self.y)

In [78]:
oak_type = TreeTypeFactory.get_tree_type(
    "Oak",
    "Green",
    "oak.png"
)

In [79]:
tree1 = Tree(10, 20, oak_type)
tree2 = Tree(100, 200, oak_type)
tree3 = Tree(500, 300, oak_type)

                    TreeType
              ┌─────────────────┐
              │ Oak             │
              │ Green           │
              │ oak.png         │
              └────────┬────────┘
                       ↑
              ┌────────┼────────┐
              │        │        │
             Tree     Tree     Tree
             (10,20)  (100,200) (500,300)

In [ ]:
# # Instead of every object carrying all its data:
# you move common data into shared objects:
# Shared heavy data
#        ↓
#  ┌─────┼─────┬─────┐
#  ↓     ↓     ↓     ↓
# Obj   Obj   Obj   Obj


In [ ]:
class TreeTypeFactory:

    _cache = {}

    @classmethod
    def get(cls, key):

        if key not in cls._cache:
            cls._cache[key] = create_object(key)

        return cls._cache[key]

In [82]:
class LLMConfig:

    def __init__(
        self,
        provider,
        model,
        temperature
    ):
        self.provider = provider
        self.model = model
        self.temperature = temperature


config1 = LLMConfig(
    "nvidia",
    "glm-5.2",
    0.1
)

config2 = LLMConfig(
    "nvidia",
    "glm-5.2",
    0.1
)

In [91]:
class LLMConfigFactory:

    _configs = {}

    @classmethod
    def get_config(
        cls,
        provider,
        model,
        temperature
    ):

        key = (
            provider,
            model,
            temperature
        )

        if key not in cls._configs:

            cls._configs[key] = LLMConfig(
                provider,
                model,
                temperature
            )

        return cls._configs[key]

config1 = LLMConfigFactory.get_config(
    "nvidia",
    "glm-5.2",
    0.1
)

config2 = LLMConfigFactory.get_config(
    "google",
    "glm-5.2",
    0.1
)
config3 = LLMConfigFactory.get_config(
    "google",
    "glm-5.2",
    0.1
)

In [92]:
LLMConfigFactory._configs

{('nvidia', 'glm-5.2', 0.1): <__main__.LLMConfig at 0x1d62ec46350>,
 ('google', 'glm-5.2', 0.1): <__main__.LLMConfig at 0x1d62ec30fd0>}

In [88]:
# Flyweight has a more specific goal:

# Share identical object instances/state among many objects to reduce memory/resource usage.

In [ ]:
print(config2 is config3)
# These are two objects containing identical data.

True


In [95]:
from dataclasses import dataclass


# Flyweight
@dataclass(frozen=True)
class TreeType:

    name: str
    color: str
    texture: str

    def draw(self, x, y):
        print(
            f"Drawing {self.name} "
            f"at ({x}, {y}) "
            f"with {self.texture}"
        )


# Flyweight Factory
class TreeTypeFactory:

    _cache = {}

    @classmethod
    def get_tree_type(
        cls,
        name,
        color,
        texture
    ):

        key = (name, color, texture)

        if key not in cls._cache:

            cls._cache[key] = TreeType(
                name,
                color,
                texture
            )

        return cls._cache[key]


# Context / object containing unique state
class Tree:

    def __init__(self, x, y, tree_type):
        self.x = x
        self.y = y
        self.tree_type = tree_type

    def draw(self):
        self.tree_type.draw(
            self.x,
            self.y
        )


# Shared Flyweight
oak = TreeTypeFactory.get_tree_type(
    "Oak",
    "Green",
    "oak.png"
)


# Different objects with different extrinsic state
tree1 = Tree(10, 20, oak)
tree2 = Tree(100, 200, oak)
tree3 = Tree(500, 300, oak)


tree1.draw()
tree2.draw()
tree3.draw()

Drawing Oak at (10, 20) with oak.png
Drawing Oak at (100, 200) with oak.png
Drawing Oak at (500, 300) with oak.png


               SHARED
          ┌──────────────┐
          │ Common data  │
          │ Configuration│
          │ Metadata     │
          └──────┬───────┘
                 ↑
       ┌─────────┼─────────┐
       │         │         │
    Object 1  Object 2  Object 3
       │         │         │
    unique     unique    unique
    state      state     state

In [96]:
# Prototype → COPY
# Singleton → ONE
# Flyweight → SHARE
# Cache     → REUSE RESULT

| Pattern       | Mental model     | Main purpose                            |
| ------------- | ---------------- | --------------------------------------- |
| **Adapter**   | 🔌 Translator    | Make incompatible interfaces compatible |
| **Decorator** | 🎁 Wrapper       | Add behavior without modifying object   |
| **Facade**    | 🚪 Front door    | Simplify a complex subsystem            |
| **Flyweight** | ♻️ Shared object | Reduce memory through object reuse      |


In [ ]:
# Adapter
# "What if interfaces don't match?"

# Decorator
# "What if I want additional behavior?"

# Facade
# "What if the system is too complicated to use?"

# Flyweight
# "What if I'm creating too many duplicate objects?"

## proxy

In [ ]:
# The simplest definition is:
# Proxy provides a substitute or representative for another object and controls access to the real object.

In [ ]:
# You
#  ↓
# Security Guard
#  ↓
# Restricted Room

In [ ]:
# Client
#   ↓
# Proxy
#   ↓
# Real Object

In [ ]:
class Database:

    def get_user(self, user_id):
        print("Fetching user from database")
        return {"id": user_id, "name": "Manish"}

db = Database()
user = db.get_user(10)

Fetching user from database


In [100]:
class Database:
    def get_user(idx):
        # authentication
        # authorization
        # logging
        # caching
        # rate limiting
        # database query
        pass

In [101]:
from abc import ABC, abstractmethod


class DatabaseInterface(ABC):

    @abstractmethod
    def get_user(self, user_id):
        pass

class Database(DatabaseInterface):

    def get_user(self, user_id):

        print("Fetching from database")

        return {
            "id": user_id,
            "name": "Manish"
        }

class DatabaseProxy(DatabaseInterface):

    def __init__(self, database):
        self.database = database

    def get_user(self, user_id):

        print("Checking access")

        return self.database.get_user(user_id)


db = Database()

proxy = DatabaseProxy(db)

user = proxy.get_user(10)

Checking access
Fetching from database


In [102]:
# db = Database()

# proxy = DatabaseProxy(db)

# user = proxy.get_user(10)

In [103]:
class DatabaseProxy(DatabaseInterface):

    def __init__(self, database, is_admin):
        self.database = database
        self.is_admin = is_admin

    def get_user(self, user_id):

        if not self.is_admin:
            raise PermissionError(
                "Access denied"
            )

        return self.database.get_user(user_id)

db = Database()

proxy = DatabaseProxy(
    db,
    is_admin=False
)

proxy.get_user(10)

PermissionError: Access denied

In [ ]:
# Client
#   ↓
# Proxy
#   ↓
# Access allowed?
#   │
#   ├── NO → Reject
#   │
#   └── YES
#         ↓
#       Real Object


# "Before you reach the real object, I will control how you access it."

In [106]:
# Authorization Proxy
#         ↓
# "Are you allowed?"

# Caching Proxy
#         ↓
# "Do I already have the result?"

# Rate Limit Proxy
#         ↓
# "Are you making too many requests?"

# Remote Proxy
#         ↓
# "Let me communicate with the remote service."

# Virtual Proxy
#         ↓
# "Should I create/load the expensive object now?"

In [109]:
# 1. Protection Proxy

# Client
#  ↓
# AuthorizationProxy
#  ↓
# RealService

# if not user.is_admin:
#     raise PermissionError()


# Client
#  ↓
# CacheProxy
#  ↓
# RealService



# Request
#  ↓
# Cache
#  │
#  ├── HIT → return cached result
#  │
#  └── MISS
#        ↓
#    Real service


# For example, suppose loading a large ML model takes 20 seconds.
# model = LargeModel()
# proxy = ModelProxy()

import time

# 1. The Heavy Object (Real Subject)
class LargeMLModel:
    def __init__(self):
        print("🤖 [SYSTEM] Starting to load massive ML weights...")
        time.sleep(3)  # Simulating a heavy 3-second delay
        print("🤖 [SYSTEM] ML Model successfully loaded into RAM!")

    def predict(self, data):
        return f"Prediction result for '{data}'"


# 2. The Virtual Proxy (The Stand-In)
class ModelProxy:
    def __init__(self):
        # We hold a reference space for the model, but it is EMPTY (None) at startup
        self._real_model = None
        print("📦 [PROXY] Proxy initialized instantly. No memory used yet.")

    def predict(self, data):
        # The core logic: "Is this the first request?"
        if self._real_model is None:
            print("⚡ [PROXY] First request received! Triggering lazy load now...")
            self._real_model = LargeMLModel()  # The expensive object is born here
        
        # Forward the request to the real model
        return self._real_model.predict(data)


# --- Application Flow ---
print("--- Step 1: App Starts ---")
proxy = ModelProxy()  # Boots instantly! No delay.

print("\n--- Step 2: Doing other app tasks ---")
print("User is browsing the UI menu...")

print("\n--- Step 3: First Request Made ---")
# This triggers the actual 3-second initialization loading sequence
print(proxy.predict("Image_01.jpg"))

print("\n--- Step 4: Second Request Made ---")
# Runs instantly because self._real_model is already loaded!
print(proxy.predict("Image_02.jpg"))


--- Step 1: App Starts ---
📦 [PROXY] Proxy initialized instantly. No memory used yet.

--- Step 2: Doing other app tasks ---
User is browsing the UI menu...

--- Step 3: First Request Made ---
⚡ [PROXY] First request received! Triggering lazy load now...
🤖 [SYSTEM] Starting to load massive ML weights...
🤖 [SYSTEM] ML Model successfully loaded into RAM!
Prediction result for 'Image_01.jpg'

--- Step 4: Second Request Made ---
Prediction result for 'Image_02.jpg'


In [ ]:
class LoggingProxy:

    def __init__(self, service):
        self.service = service

    def get_user(self, user_id):

        print(f"Calling get_user({user_id})")

        result = self.service.get_user(user_id)

        print("Call completed")

        return result

In [ ]:
# # Decorator

# # The primary purpose is:

# # Add behavior.


# Proxy

# The primary purpose is:

# Control access to the object.

In [ ]:
class LLMProxy:

    def __init__(self, llm, user):
        self.llm = llm
        self.user = user

    def generate(self, prompt):

        if not self.user.is_allowed:
            raise PermissionError(
                "User cannot access LLM"
            )

        return self.llm.generate(prompt)

llm = RealLLM()

llm = LLMProxy(
    llm,
    user
)

response = llm.generate(
    "Analyze this company"
)

In [ ]:
class RateLimitProxy:

    def __init__(self, llm, rate_limiter):
        self.llm = llm
        self.rate_limiter = rate_limiter

    def generate(self, prompt):

        if not self.rate_limiter.allow():
            raise RuntimeError(
                "Rate limit exceeded"
            )

        return self.llm.generate(prompt)

In [ ]:
class CachingLLMProxy:

    def __init__(self, llm):
        self.llm = llm
        self.cache = {}

    def generate(self, prompt):

        if prompt in self.cache:

            print("Cache hit")

            return self.cache[prompt]

        print("Cache miss")

        response = self.llm.generate(prompt)

        self.cache[prompt] = response

        return response

llm = RealLLM()

llm = CachingLLMProxy(llm)

In [ ]:
class LargeModel:

    def __init__(self):
        print("Loading huge model...")

class ModelProxy:

    def __init__(self):
        self.model = None

    def predict(self, input_data):

        if self.model is None:
            print("Loading model...")
            self.model = LargeModel()

        return self.model.predict(input_data)

model = ModelProxy()

model.predict(data)

In [110]:
                #         Agent
                #           ↓
                #    AuthorizationProxy
                #           ↓
                #     RateLimitProxy
                #           ↓
                #      CacheProxy
                #           ↓
                #    LoggingProxy
                #           ↓
                #        LLM

In [111]:
from abc import ABC, abstractmethod


# Subject
class PaymentService(ABC):

    @abstractmethod
    def pay(self, amount):
        pass


# Real Subject
class RealPaymentService(PaymentService):

    def pay(self, amount):
        print(f"Processing payment: ₹{amount}")


# Proxy
class PaymentProxy(PaymentService):

    def __init__(self, payment_service, is_authenticated):
        self.payment_service = payment_service
        self.is_authenticated = is_authenticated

    def pay(self, amount):

        print("Checking authentication...")

        if not self.is_authenticated:
            raise PermissionError(
                "User is not authenticated"
            )

        print("Access granted")

        return self.payment_service.pay(amount)

| Pattern       | Mental model     | Main purpose                            |
| ------------- | ---------------- | --------------------------------------- |
| **Adapter**   | 🔌 Translator    | Make incompatible interfaces compatible |
| **Decorator** | 🎁 Wrapper       | Add behavior                            |
| **Facade**    | 🚪 Front door    | Simplify a complex subsystem            |
| **Flyweight** | ♻️ Shared object | Reduce memory through sharing           |
| **Proxy**     | 🛡️ Gatekeeper   | Control access to an object             |
